In [ ]:
import sqlite3                                 
import pandas as pd                             
 
con = sqlite3.connect('../data/raw/airtel.db')  
 
def q(sql):                                     
    return pd.read_sql(sql, con)

In [33]:
q("""SELECT name FROM sqlite_master               
WHERE  type = 'table'                          
ORDER  BY name;""")

,name
0,circles
1,complaints
2,customers
3,plans
4,recharges
5,usage_daily


In [34]:
q("""SELECT 'circles' AS tbl, COUNT(*) AS n FROM circles
UNION ALL SELECT 'plans',         COUNT(*) FROM plans
UNION ALL SELECT 'customers',     COUNT(*) FROM customers
UNION ALL SELECT 'usage_daily',   COUNT(*) FROM usage_daily
UNION ALL SELECT 'recharges',     COUNT(*) FROM recharges
UNION ALL SELECT 'complaints',    COUNT(*) FROM complaints; """)

,tbl,n
0,circles,22
1,plans,14
2,customers,22000
3,usage_daily,1326739
4,recharges,408089
5,complaints,26841


In [48]:
import sqlite3                                 
import pandas as pd                             
 
pcon = sqlite3.connect('../data/practice/practice.db')  
pd.read_sql("select * from loans",pcon)
def pq(sql):                                     
    return pd.read_sql(sql, pcon)

In [36]:
pq("""select * from shop_customers;""")

,customer_id,name,city,joined_on
0,1,Asha,Kochi,2024-01-15
1,2,Ravi,Chennai,2024-03-02
2,3,Meera,Kochi,2024-06-20
3,4,Arjun,Pune,2024-09-10
4,5,Fatima,Chennai,2025-01-05
5,6,John,Pune,2025-02-14


In [37]:
pq("""select *  from shop_orders""")


,order_id,customer_id,order_date,amount
0,101,1,2025-01-03,450
1,102,1,2025-01-20,300
2,103,2,2025-01-11,1200
3,104,3,2025-02-02,250
4,105,1,2025-02-15,600
5,106,4,2025-02-21,900
6,107,2,2025-03-05,150
7,108,5,2025-03-09,700
8,109,3,2025-03-18,400
9,110,2,2025-03-28,350


In [38]:
pq("""select *  from members""")


,member_id,name,joined_on
0,1,Anil,2023-07-01
1,2,Divya,2024-02-10
2,3,Karthik,2024-05-25
3,4,Lakshmi,2024-11-30
4,5,Nisha,2025-01-12
5,6,Sameer,2025-03-01


In [39]:
pq("""select *  from loans""")

,loan_id,member_id,book,loan_date,returned_on
0,1,1,Wings of Fire,2025-01-04,2025-01-18
1,2,2,Godan,2025-01-09,2025-01-30
2,3,1,Gitanjali,2025-01-22,2025-02-05
3,4,3,Malgudi Days,2025-02-03,NaN
4,5,2,The Guide,2025-02-11,2025-02-25
5,6,4,Train to Pakistan,2025-02-19,2025-03-10
6,7,1,Discovery of India,2025-03-02,NaN
7,8,5,Ignited Minds,2025-03-06,2025-03-20
8,9,2,Wise and Otherwise,2025-03-15,NaN
9,10,3,Swami and Friends,2025-03-27,2025-04-08


In [40]:
pq("""select customer_id,
        count(*) as orders,
        sum(case when amount >=500
        then 1
        else 0
        end)as greater
        from shop_orders
        group by customer_id""")

,customer_id,orders,greater
0,1,3,1
1,2,3,1
2,3,2,0
3,4,1,1
4,5,1,1


In [41]:
pq("""
    select count(*)as orders,
    sum(amount>=500) as big_orders,
    round(100.0*avg(amount>=500),2)as perc
    from shop_orders;
    """)

,orders,big_orders,perc
0,10,4,40.0


In [42]:
pq("""
    select name,
    joined_on,
    cast(julianday('2025-03-31')-julianday(joined_on)as int)as  days
    from shop_customers;
    """)

,name,joined_on,days
0,Asha,2024-01-15,441
1,Ravi,2024-03-02,394
2,Meera,2024-06-20,284
3,Arjun,2024-09-10,202
4,Fatima,2025-01-05,85
5,John,2025-02-14,45


In [59]:
#How many orders and how much sales in each month ?
#month ,order, sales 
#shop_orders

pq("""SELECT strftime('%Y-%m', order_date) AS month,   
       COUNT(*)    AS orders,                  
       SUM(amount) AS sales                      
FROM   shop_orders                           
GROUP  BY month ORDER BY month; """)

,month,orders,sales
0,2025-01,3,1950
1,2025-02,3,1750
2,2025-03,4,1600


In [ ]:
pq("""
    select c.name,c.city
    from shop_customers as c
    left join shop_orders as o
        on o.customer_id = c.customer_id
    where o.customer_id is null;
    """)

,name,city
0,John,Pune


In [44]:
pq("""
    select m.name
    from members as m
    left join shop_orders as l
        on l.customer_id = m.member_id
    where l.customer_id is null;
""")

,name
0,Sameer


In [45]:
pq("""
with s as (
        select customer_id,
                   sum(amount) as spend
        from shop_orders group by customer_id)
select c.name,s.spend,
           row_number() over (order by s.spend desc) as rn,
           ntile(2) over (order by s.spend desc) as half
from s join shop_customers c
  on c.customer_id = s.customer_id
order by s.spend desc;
""")

,name,spend,rn,half
0,Ravi,1700,1,1
1,Asha,1350,2,1
2,Arjun,900,3,1
3,Fatima,700,4,2
4,Meera,650,5,2


In [49]:
def run_sql_file(path):
    with open(path, 'r') as f:
        script = f.read()
    pcon.executescript(script)
    pcon.commit()
    print('ran', path)

In [51]:
run_sql_file('../sql/practice/index.sql') 

ran ../sql/practice/index.sql


In [52]:
pq("""select name, tbl_name from sqlite_master
where type = 'index'; """)

,name,tbl_name
0,ix_orders_customer,shop_orders


In [54]:
pq("""
SELECT * FROM shop_orders
WHERE customer_id=2
""")
pq("""
SELECT * FROM shop_orders
WHERE customer_id=2
""")

,order_id,customer_id,order_date,amount
0,103,2,2025-01-11,1200
1,107,2,2025-03-05,150
2,110,2,2025-03-28,350


In [57]:
pq("""
SELECT strftime('%m','2026-10-28')as month;
""")

,month
0,10


In [58]:
pq("""
SELECT julianday('Sinan') as id;
""")

,id
0,None


In [61]:
pq("""select sum(is_churned = 1 and churn_date is not null)
        as left_dated,
      sum(is_churned = 0 and churn_date is null)
          as stayed_blank
      as stayed_blank
from customers;""")

DatabaseError: Execution failed on sql 'select sum(is_churned = 1 and churn_date is not null)
        as left_dated,
      sum(is_churned = 0 and churn_date is null)
          as stayed_blank
      as stayed_blank
from customers;': near "as": syntax error